# PSS Enrichment with CKN — Tutorial

Walkthrough of the `skm_tools.enrich_pss` module. Three parts:

- **PART ONE** — Build (or load) an enriched PSS network: every CKN node/edge that touches PSS is added, tagged `origin="CKN"`; PSS itself is untouched and always tagged `origin="PSS"`.
- **PART TWO** — Inspect what the enrichment introduced:
  - **Export A** — PSS pairs now directly connected by a new CKN edge
  - **Export B** — PSS pairs now connected through exactly one CKN intermediate
- **PART THREE** — Pick any node (random or chosen) and check how much CKN enriched its immediate neighbourhood.

Run this notebook top to bottom in VS Code (with the Jupyter extension) using a kernel that has `networkx`, `pandas`, and (for the visualization cells) `py4cytoscape` with Cytoscape running locally.

## Setup

In [24]:
from pathlib import Path
from datetime import date
import importlib
import networkx as nx
import pandas as pd

from skm_tools import load_networks, enrich_pss, ckn_utils
importlib.reload(enrich_pss)   # pick up edits without restarting the kernel

base_dir = Path("./")
data_dir = base_dir / "data"
output_dir = base_dir / "output"
data_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print("Imports OK")

Imports OK


---
## PART ONE — Build (or load) the enriched PSS network

`build_enriched_pss(pss, ckn)`:
1. Copies PSS as-is (`origin="PSS"` on every node/edge)
2. Adds every CKN edge that touches at least one PSS node, and the CKN node(s) it brings in (`origin="CKN"`) — unless PSS already has an edge between the same two nodes (PSS always wins)
3. Adds CKN-CKN edges between two CKN nodes that were both pulled in by step 2 (keeps local CKN connectivity intact, needed for Export B and for node-neighbourhood inspection in PART THREE)

No rank filter is applied here — every CKN rank is kept, so the saved network is reusable for any rank threshold later.

In [29]:
# ---- USER INPUT ------------------------------------------------------------
ENRICHED_DIR = output_dir / "enriched_pss"
REBUILD = True   # set True to force a rebuild even if saved files exist
# ---------------------------------------------------------------------------

today = date.today().isoformat()

pss = load_networks.pss_dinar_to_networkx(
    edge_path=data_dir / f"pss-dinar-edges-public-{today}.tsv",
    node_path=data_dir / f"pss-dinar-nodes-public-{today}.tsv",
)
print(f"PSS: {pss.number_of_nodes():,} nodes, {pss.number_of_edges():,} edges")

graphml_path = ENRICHED_DIR / "enriched.graphml"
if graphml_path.exists() and not REBUILD:
    print(f"\nFound saved enriched graph at {graphml_path} -- loading.")
    enriched = enrich_pss.load_enriched(ENRICHED_DIR)
else:
    print("\nBuilding enriched graph from scratch ...")
    ckn = load_networks.ckn_to_networkx(
        edge_path=data_dir / "AtCKN-v2-2023.06.tsv.gz",
        node_path=data_dir / "AtCKN-v2-2023.06_node-annot.tsv.gz",
        add_reciprocal_edges=True,
    )
    ckn_utils.filter_ckn_nodes(ckn, species=['ath'])
    print(f"CKN: {ckn.number_of_nodes():,} nodes, {ckn.number_of_edges():,} edges")

    enriched = enrich_pss.build_enriched_pss(pss, ckn, verbose=True)
    enrich_pss.save_enriched(enriched, ENRICHED_DIR)

print(f"\nEnriched: {enriched.number_of_nodes():,} nodes, "
      f"{enriched.number_of_edges():,} edges")

Attempting to download the edge list to data\pss-dinar-edges-public-2026-07-02.tsv. Success.
	Attempting to download the node annotations (PSS) to data\pss-dinar-nodes-public-2026-07-02.tsv.other.tmp. Success.
	Attempting to download the node annotations (CKN) to data\pss-dinar-nodes-public-2026-07-02.tsv.genes.tmp Success.
Creating node annotation file... Success.
PSS: 1,071 nodes, 4,395 edges

Building enriched graph from scratch ...
Removed 162 nodes from network.
CKN: 26,072 nodes, 898,049 edges
[1/3] Copying PSS nodes and edges …
      PSS: 1,071 nodes, 4,395 edges
[2/3] Adding CKN edges that touch PSS …
      + 11,917 CKN nodes, 153,183 CKN edges (touching PSS)
[3/3] Adding CKN edges between pulled-in CKN nodes …
      + 557,277 CKN-CKN edges
----------------------------------------------------------
  Enriched: 12,988 nodes (1,071 PSS + 11,917 CKN)
  Enriched: 714,855 edges (4,395 PSS + 710,460 CKN)
----------------------------------------------------------
OK: Saved enriched ne

### Before / after summary

In [30]:
_ = enrich_pss.print_enrichment_summary(pss, enriched)

                          PSS (before)   Enriched (after)
--------------------------------------------------
                   Nodes         1,071             12,988
      of which CKN-added             —             11,917
                   Edges         4,395            714,855
      of which CKN-added             —            710,460


### Visualize a small sample in Cytoscape

The full enriched network is too large to render usefully. We take the
ego-graph (radius 1) around a few PSS nodes to visually confirm PSS
(blue) vs CKN (orange) colouring.

In [31]:
pss_nodes = [n for n, d in enriched.nodes(data=True) if d.get("origin") == "PSS"]
sample_seed_nodes = pss_nodes[:2]

sample_nodes = set(sample_seed_nodes)
for n in sample_seed_nodes:
    sample_nodes |= set(nx.ego_graph(enriched.to_undirected(), n, radius=1).nodes())

sample = enriched.subgraph(sample_nodes).copy()
print(f"Sample subgraph: {sample.number_of_nodes()} nodes, {sample.number_of_edges()} edges")

suid_sample = enrich_pss.visualize_enriched_network(sample, network_name="enriched_pss_sample")

Sample subgraph: 114 nodes, 1254 edges
Sending to Cytoscape: 114 nodes, 1254 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 7614
OK: Network 'enriched_pss_sample' created in Cytoscape (SUID: 7614)


---
## PART TWO — Enrichment analysis

Set the rank threshold applied here (analysis time, not build time).

In [35]:
# ---- USER INPUT ------------------------------------------------------------
MAX_RANK = 1   # 0 = highest confidence only, None = keep all ranks
# ---------------------------------------------------------------------------

### Export A — new direct CKN edges between PSS pairs

In [36]:
export_a = enrich_pss.find_new_direct_edges(enriched, pss, max_rank=MAX_RANK)
print(f"Export A: {len(export_a):,} new direct-edge PSS pairs (rank <= {MAX_RANK})")
export_a.head(20)

Export A: 830 new direct-edge PSS pairs (rank <= 1)


,u,v,rank,direction
0,AT1G08320,AT2G14580,0,u->v
1,AT1G08320,AT2G14610,0,u->v
2,AT1G08450,AT1G04310,0,both
3,AT1G08450,AT1G66340,0,both
4,AT1G08450,AT2G40940,0,both
5,AT1G08450,AT3G04580,0,both
6,AT1G08450,AT3G23150,0,both
7,AT1G09210,AT1G04310,0,both
8,AT1G09210,AT1G66340,0,both
9,AT1G09210,AT2G40940,0,both


In [ ]:
AT2G26150, AT3G12580

NameError: name 'AT2G26150' is not defined

In [ ]:
export_a.to_csv(output_dir / "export_A_new_direct_edges.csv", index=False)
print("Saved -> output/export_A_new_direct_edges.csv")

Saved -> output/export_A_new_direct_edges.csv


### Export B — PSS pairs via exactly one CKN intermediate

In [47]:
export_b = enrich_pss.find_new_intermediate_pairs(enriched, pss, max_rank=MAX_RANK)
n_unique_pairs = export_b[["AT2G26150", "AT3G12580"]].drop_duplicates().shape[0] if len(export_b) else 0
print(f"Export B: {len(export_b):,} pair-intermediate triples, {n_unique_pairs:,} unique pairs (rank <= {MAX_RANK})")
export_b.head(20)

KeyError: "None of [Index(['AT2G26150', 'AT3G12580'], dtype='str')] are in the [columns]"

In [39]:
export_b.to_csv(output_dir / "export_B_one_intermediate_pairs.csv", index=False)
print("Saved -> output/export_B_one_intermediate_pairs.csv")

Saved -> output/export_B_one_intermediate_pairs.csv


### Inspect one example pair from each export in Cytoscape

In [40]:
# ---- USER INPUT ------------------------------------------------------------
PAIR_A_INDEX = 0
PAIR_B_INDEX = 0
MAX_NEIGHBOURS = 10
# ---------------------------------------------------------------------------

In [41]:
if len(export_a):
    row_a = export_a.iloc[PAIR_A_INDEX]
    print(f"Export A pair: {row_a.u} <-> {row_a.v}  (rank {row_a['rank']}, direction {row_a.direction})")

    sub_a = enrich_pss.extract_pair_subnetwork(
        enriched, pss, a=row_a.u, b=row_a.v, intermediate=None,
        max_neighbours=MAX_NEIGHBOURS,
    )
    print(f"  Subnetwork: {sub_a.number_of_nodes()} nodes, {sub_a.number_of_edges()} edges")
    suid_a = enrich_pss.visualize_pair_subnetwork(sub_a, network_name=f"expA_{row_a.u}_{row_a.v}")
else:
    print("Export A is empty at this rank threshold.")

Export A pair: AT1G08320 <-> AT2G14580  (rank 0, direction u->v)
  Subnetwork: 22 nodes, 133 edges
Sending to Cytoscape: 22 nodes, 133 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 14712
OK: Network 'expA_AT1G08320_AT2G14580' created in Cytoscape (SUID: 14712)


In [42]:
if len(export_b):
    row_b = export_b.iloc[PAIR_B_INDEX]
    print(f"Export B triple: {row_b.u} <- {row_b.intermediate} -> {row_b.v}  "
          f"(rank_ux={row_b.rank_ux}, rank_xv={row_b.rank_xv}, path={row_b.path_rank})")

    sub_b = enrich_pss.extract_pair_subnetwork(
        enriched, pss, a=row_b.u, b=row_b.v, intermediate=row_b.intermediate,
        max_neighbours=MAX_NEIGHBOURS,
    )
    print(f"  Subnetwork: {sub_b.number_of_nodes()} nodes, {sub_b.number_of_edges()} edges")
    suid_b = enrich_pss.visualize_pair_subnetwork(
        sub_b, network_name=f"expB_{row_b.u}_{row_b.intermediate}_{row_b.v}")
else:
    print("Export B is empty at this rank threshold.")

Export B triple: AT1G04100 <- AUX/IAA|WRKY57 -> AT1G04240  (rank_ux=0, rank_xv=0, path=0)
  Subnetwork: 9 nodes, 37 edges
Sending to Cytoscape: 9 nodes, 37 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 15669
OK: Network 'expB_AT1G04100_AUX/IAA|WRKY57_AT1G04240' created in Cytoscape (SUID: 15669)


---
## PART THREE — Node neighbourhood inspection

Pick any node (random, or one you choose) and check how much CKN enriched
its immediate neighbourhood.

**How big should the neighbourhood be?**
- **`radius=1`** (default, recommended) — direct neighbours only. Answers
  "did CKN change who this exact node talks to?" Always small enough to
  read directly.
- **`radius=2`** — neighbours-of-neighbours. Useful for spotting pathway
  crosstalk opened up by a CKN intermediate, but some CKN hub nodes bridge
  hundreds to thousands of PSS pairs (see PART TWO Export B), so a radius-2
  view can blow up. Always combine with `max_neighbours` (try 15-25) when
  using radius 2.
- **`radius>=3`** is not recommended for visually inspecting a single node
  — use the Export A/B tables from PART TWO for that instead.

In [43]:
# ---- USER INPUT ------------------------------------------------------------
NODE = None        # set to a specific AGI code / CKN id, or leave None for random
NODE_TYPE = "PSS"  # "PSS", "CKN", or None -- only used when NODE is None
SEED = None         # set an int for a reproducible random pick
RADIUS = 1
MAX_NEIGHBOURS = 20  # recommended whenever RADIUS > 1
# ---------------------------------------------------------------------------

node = NODE or enrich_pss.pick_random_node(enriched, node_type=NODE_TYPE, seed=SEED)
print(f"Inspecting node: {node}  (origin={enriched.nodes[node].get('origin')})")

Inspecting node: AT3G25070  (origin=PSS)


In [44]:
stats = enrich_pss.get_node_neighbourhood_enrichment(
    pss, enriched, node, radius=RADIUS, max_neighbours=MAX_NEIGHBOURS
)
enrich_pss.print_node_neighbourhood_summary(stats)

Node: AT3G25070  (radius=1)
  PSS neighbours (before)      : 3
  Enriched neighbours (after)  : 20
  of which new from CKN        : 14 (70.0%)


In [45]:
neighbourhood = enrich_pss.extract_node_neighbourhood(
    enriched, node, radius=RADIUS, max_neighbours=MAX_NEIGHBOURS
)
suid_node = enrich_pss.visualize_node_neighbourhood(
    neighbourhood, node, network_name=f"neighbourhood_{node}_r{RADIUS}"
)

Sending to Cytoscape: 21 nodes, 124 edges …
Applying default style...
Applying preferred layout
Applied CKN-default to 16056
OK: Network 'neighbourhood_AT3G25070_r1' created in Cytoscape (SUID: 16056)


---
## Summary

In [16]:
s = enrich_pss.get_enrichment_summary(pss, enriched)
n_unique_pairs = export_b[["u", "v"]].drop_duplicates().shape[0] if len(export_b) else 0

print(f"""
==================================================================
  PSS enrichment summary  (rank <= {MAX_RANK})
==================================================================

  Enriched graph
    Nodes : {s['enriched_nodes']:>7,}  ({s['pss_nodes']:,} PSS + {s['ckn_nodes_added']:,} CKN)
    Edges : {s['enriched_edges']:>7,}  ({s['pss_edges']:,} PSS + {s['ckn_edges_added']:,} CKN)

  Export A -- new direct CKN edges between PSS pairs
    {len(export_a):>7,} pairs -> output/export_A_new_direct_edges.csv

  Export B -- 1-CKN-intermediate pairs
    {len(export_b):>7,} (pair, intermediate) triples
    {n_unique_pairs:>7,} unique PSS pairs -> output/export_B_one_intermediate_pairs.csv

  Part Three -- inspected node
    {stats['node']}  radius={stats['radius']}
    {stats['ckn_added_neighbours']} of {stats['enriched_neighbours']} neighbours are new from CKN
    ({stats['pct_new']:.1f}%)

  Files in {ENRICHED_DIR}/
    enriched.graphml, enriched_nodes.tsv, enriched_edges.tsv, enrichment_diff.tsv
==================================================================
""")


  PSS enrichment summary  (rank <= 1)

  Enriched graph
    Nodes :  13,024  (1,071 PSS + 11,953 CKN)
    Edges : 715,125  (4,395 PSS + 710,730 CKN)

  Export A -- new direct CKN edges between PSS pairs
        836 pairs -> output/export_A_new_direct_edges.csv

  Export B -- 1-CKN-intermediate pairs
      6,586 (pair, intermediate) triples
      4,498 unique PSS pairs -> output/export_B_one_intermediate_pairs.csv

  Part Three -- inspected node
    AT4G31800  radius=1
    15 of 20 neighbours are new from CKN
    (75.0%)

  Files in output\enriched_pss/
    enriched.graphml, enriched_nodes.tsv, enriched_edges.tsv, enrichment_diff.tsv

